# Práctica de análisis: Alumbrado público adaptativo
## Modelos Matemáticos y Simulación — Ingeniería en Software — Grupo 10

**Entorno:** Google Colab (`numpy`, `pandas`, `matplotlib`, `networkx`)

**Datos:** archivos CSV exportados desde la app (pestaña **Análisis → Exportar datos**).

### Objetivos de aprendizaje
Al finalizar este cuaderno el grupo será capaz de:
1. Ajustar el **modelo de consumo** $P = a_0 + a_1 \cdot brillo$ por **mínimos cuadrados** con los datos medidos por el INA219 y evaluar el ajuste con $r^2$ y $s_{y/x}$.
2. Calcular la **energía** de cada noche resolviendo $\frac{dE}{dt} = P(t)$ con el **método de Euler** y compararla con la que calculó la app.
3. Obtener con **Dijkstra** las distancias desde cada entrada y construir el plan de la **ola de luz**.
4. Describir la **máquina de estados** de cada luminaria a partir de las lecturas reales.
5. Comparar las estrategias **permanente, programada y adaptativa** con datos de la maqueta y de la simulación.

### Estructura
| Sección | Modelo | Archivo que usa |
|---|---|---|
| 1 | Carga de datos | los 4 CSV |
| 2 | Modelo 2a: regresión lineal del consumo | `calibracion.csv` |
| 3 | Modelo 2b: energía por el método de Euler | `lecturas_maqueta.csv`, `simulacion_por_minuto.csv` |
| 4 | Modelo 3: grafo de calles y Dijkstra | (se construye aquí) |
| 5 | Modelo 1: máquina de estados | `lecturas_maqueta.csv` |
| 6 | Comparación de estrategias | `resumen_noches.csv`, `simulacion_por_minuto.csv` |

> **Instrucciones:** ejecuta las celdas en orden. Si una sección no tiene datos todavía (por ejemplo, no se ha calibrado la maqueta), la celda lo indica y se puede volver a ejecutar después de exportar de nuevo.

In [ ]:
# Librerías que usaremos a lo largo de todo el cuaderno
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import display

plt.rcParams['figure.figsize'] = (7, 4)
COLORES = {'permanent': '#8e8e93', 'scheduled': '#5e5ce6', 'adaptive': '#ffb340'}
NOMBRES = {'permanent': 'Permanente', 'scheduled': 'Programada', 'adaptive': 'Adaptativa'}
print('networkx versión', nx.__version__)

### Funciones de apoyo
Son las mismas funciones de las prácticas de clase: `tabla_sumatorias`, `reg_lin` y `metricas` (Práctica 7), el método de **Euler** (Práctica 2) y `dijkstra_tabla` con `reconstruir` (Práctica 9, TPE4 del grupo). Cada una implementa una definición de la teoría.

In [ ]:
def tabla_sumatorias(x, y):
    # Construye la tabla de sumatorias (igual que en la hoja de cálculo) y la muestra.
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    tabla = pd.DataFrame({'x': x, 'y': y, 'xy': x * y, 'x^2': x**2, 'y^2': y**2},
                         index=pd.RangeIndex(1, len(x) + 1, name='n'))
    sumas = tabla.sum()
    tabla.loc['Σ'] = sumas
    display(tabla)
    return sumas


def reg_lin(x, y):
    # Regresión lineal por mínimos cuadrados con las fórmulas de las sumatorias.
    # Devuelve a0 (intercepto), a1 (pendiente) y r^2.
    X = np.asarray(x, dtype=float)
    Y = np.asarray(y, dtype=float)
    n = len(X)
    Sx, Sy = X.sum(), Y.sum()
    Sxy, Sxx, Syy = (X * Y).sum(), (X**2).sum(), (Y**2).sum()
    a1 = (n * Sxy - Sx * Sy) / (n * Sxx - Sx**2)
    a0 = (Sxx * Sy - Sxy * Sx) / (n * Sxx - Sx**2)
    r2 = ((n * Sxy - Sx * Sy) / (np.sqrt(n * Sxx - Sx**2) * np.sqrt(n * Syy - Sy**2)))**2
    return a0, a1, r2


def metricas(y, y_est, p=2):
    # St, Sr, r^2 y error estándar s_y/x. p = número de parámetros del modelo.
    y = np.asarray(y, dtype=float)
    y_est = np.asarray(y_est, dtype=float)
    St = np.sum((y - y.mean())**2)
    Sr = np.sum((y - y_est)**2)
    r2 = (St - Sr) / St
    syx = np.sqrt(Sr / (len(y) - p)) if len(y) > p else np.nan
    return {'St': St, 'Sr': Sr, 'r2': r2, 'syx': syx}


def energia_euler(tiempos, potencias):
    # Método de Euler para dE/dt = P(t):  E(k+1) = E(k) + P(k) · Δt
    # tiempos en segundos, potencias en mW (o W). Devuelve la energía acumulada en mWh (o Wh).
    energia = [0.0]
    for k in range(len(tiempos) - 1):
        dt = tiempos[k + 1] - tiempos[k]
        energia.append(energia[-1] + potencias[k] * dt)
    return np.array(energia) / 3600


def dijkstra_tabla(G, origen):
    # Dijkstra con la tabla de iteraciones (igual que en la práctica de grafos del grupo).
    D = {v: np.inf for v in G}
    pred = {v: None for v in G}
    D[origen] = 0
    visitados, iteraciones = [], []
    while len(visitados) < len(G):
        u = min((v for v in G if v not in visitados), key=lambda v: D[v])
        visitados.append(u)
        for v in G.neighbors(u):
            if v not in visitados and D[u] + G[u][v]['weight'] < D[v]:
                D[v] = D[u] + G[u][v]['weight']
                pred[v] = u
        fila = {'Iteración': len(visitados), 'Visitado': u}
        fila.update({v: ('✔ ' if v in visitados else '') + ('∞' if D[v] == np.inf else f'{D[v]:.0f}') for v in G})
        iteraciones.append(fila)
    return D, pred, pd.DataFrame(iteraciones).set_index('Iteración')


def reconstruir(pred, destino):
    camino = [destino]
    while pred[camino[-1]] is not None:
        camino.append(pred[camino[-1]])
    return camino[::-1]


def hay_datos(tabla, mensaje):
    # Evita errores cuando todavía no hay datos de la maqueta.
    if len(tabla) == 0:
        print(mensaje)
        return False
    return True


print('Funciones listas: tabla_sumatorias, reg_lin, metricas, energia_euler, dijkstra_tabla, reconstruir')

---
## Sección 1 — Carga de datos

En la app, abre la pestaña **Análisis** y en **Exportar datos** descarga los cuatro archivos:

| Archivo | Contenido |
|---|---|
| `resumen_noches.csv` | Una fila por noche (maqueta o simulación): estrategia, personas, energía, potencia media |
| `lecturas_maqueta.csv` | Todas las lecturas del ESP32 cada 0,5 s de las noches registradas |
| `calibracion.csv` | Brillo (%) y potencia medida (mW) de la calibración |
| `simulacion_por_minuto.csv` | Potencia media de cada minuto de cada noche simulada |

En Colab, la celda siguiente pide subirlos. En una computadora, colócalos en la misma carpeta que el cuaderno.

In [ ]:
ARCHIVOS = ['resumen_noches.csv', 'lecturas_maqueta.csv', 'calibracion.csv', 'simulacion_por_minuto.csv']
faltan = [archivo for archivo in ARCHIVOS if not os.path.exists(archivo)]
if faltan:
    try:
        from google.colab import files
        print('Sube los archivos exportados desde la app:', faltan)
        files.upload()
    except ImportError:
        print('Copia estos archivos en la carpeta del cuaderno:', faltan)

resumen = pd.read_csv('resumen_noches.csv')
lecturas = pd.read_csv('lecturas_maqueta.csv')
calibracion = pd.read_csv('calibracion.csv')
simulacion = pd.read_csv('simulacion_por_minuto.csv')
print(f'Noches registradas: {len(resumen)}   lecturas de la maqueta: {len(lecturas)}   puntos de calibración: {len(calibracion)}   noches simuladas: {simulacion.shape[1] - 2}')
display(resumen)

---
## Sección 2 — Modelo 2a: regresión lineal del consumo

**Modelo:** $\;P = a_0 + a_1 \cdot brillo$

Durante la calibración el ESP32 sube los 6 LEDs de 0 a 100 % en pasos de 10 % y el INA219 mide la potencia. Con esos puntos ajustamos la recta por **mínimos cuadrados**:

$$a_1 = \frac{n S_{xy} - S_x S_y}{n S_{xx} - S_x^2} \qquad a_0 = \frac{S_{xx} S_y - S_{xy} S_x}{n S_{xx} - S_x^2}$$

y medimos qué tan bueno es el ajuste con $r^2$ y el error estándar $s_{y/x}$.

In [ ]:
if hay_datos(calibracion, 'Aún no hay calibración: en la app, pestaña Monitoreo, pulsa "Calibrar consumo" y vuelve a exportar.'):
    x = calibracion['brightness'].to_numpy(dtype=float)
    y = calibracion['powerMw'].to_numpy(dtype=float)
    sumas = tabla_sumatorias(x, y)
    a0, a1, r2 = reg_lin(x, y)
    ajuste = metricas(y, a0 + a1 * x)
    print(f'Modelo:  P = {a0:.3f} + {a1:.4f} · brillo   (mW, brillo en %)')
    print(f'r² = {r2:.5f}   s_y/x = {ajuste["syx"]:.3f} mW')
    print('Verificación np.polyfit [a1, a0]:', np.round(np.polyfit(x, y, 1), 5))

    fig, (ax, axr) = plt.subplots(1, 2, figsize=(12, 4))
    xx = np.linspace(0, 100, 200)
    ax.scatter(x, y, color='black', zorder=3, label='Medido (INA219)')
    ax.plot(xx, a0 + a1 * xx, lw=2, label='Recta ajustada')
    ax.set_title('Consumo de los LEDs'); ax.set_xlabel('Brillo (%)'); ax.set_ylabel('Potencia (mW)')
    ax.grid(alpha=0.3); ax.legend()
    axr.axhline(0, color='gray', lw=1)
    axr.stem(x, y - (a0 + a1 * x))
    axr.set_title('Residuos  e = y − ŷ'); axr.set_xlabel('Brillo (%)'); axr.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

**Preguntas de análisis — Sección 2**
1. Interpreta $a_1$: ¿cuántos mW consume la maqueta por cada punto porcentual de brillo? ¿Cuánto consumiría un solo LED al 100 %?
2. ¿Tiene sentido físico el valor de $a_0$? ¿Qué representa el consumo con brillo 0?
3. ¿El $r^2$ justifica usar un modelo **lineal**? ¿Los residuos muestran algún patrón?

---
## Sección 3 — Modelo 2b: energía por el método de Euler

La energía es la integral de la potencia en el tiempo:

$$\frac{dE}{dt} = P(t), \qquad E(0) = 0$$

Con el **método de Euler** y las lecturas del ESP32 (una cada ~0,5 s):

$$E_{k+1} = E_k + P_k \, \Delta t_k$$

Es exactamente lo que hace la app para calcular la energía de cada noche. Aquí lo verificamos con los datos crudos.

In [ ]:
if hay_datos(lecturas, 'Aún no hay noches de la maqueta: tapa el LDR durante un minuto con la maqueta conectada y vuelve a exportar.'):
    lecturas['time'] = pd.to_datetime(lecturas['time'])
    filas = []
    for noche, datos in lecturas.groupby('night'):
        datos = datos[datos['mode'] == 'auto']
        t = (datos['time'] - datos['time'].iloc[0]).dt.total_seconds().to_numpy()
        energia = energia_euler(t, datos['powerMw'].to_numpy(dtype=float))
        filas.append({'Noche': noche, 'Estrategia': NOMBRES.get(datos['strategy'].iloc[0], '—'),
                      'Duración (s)': round(t[-1], 1), 'Energía Euler (mWh)': round(energia[-1], 3)})
    tabla_euler = pd.DataFrame(filas)
    app = resumen[resumen['source'] == 'maqueta'][['startedAt', 'energy']].rename(columns={'startedAt': 'Noche', 'energy': 'Energía app (mWh)'})
    display(tabla_euler.merge(app, on='Noche', how='left'))

    ultima = lecturas[lecturas['night'] == lecturas['night'].iloc[-1]]
    t = (ultima['time'] - ultima['time'].iloc[0]).dt.total_seconds().to_numpy()
    fig, (ax, axe) = plt.subplots(1, 2, figsize=(12, 4))
    ax.plot(t, ultima['powerMw'], color='#ffb340')
    ax.set_title('Potencia medida en la última noche'); ax.set_xlabel('Tiempo (s)'); ax.set_ylabel('mW'); ax.grid(alpha=0.3)
    axe.plot(t, energia_euler(t, ultima['powerMw'].to_numpy(dtype=float)), color='#5e5ce6')
    axe.set_title('Energía acumulada (Euler)'); axe.set_xlabel('Tiempo (s)'); axe.set_ylabel('mWh'); axe.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

En la **simulación** cada fila de `simulacion_por_minuto.csv` es la potencia media de un minuto, así que $\Delta t = 60$ s:

In [ ]:
columnas = simulacion.columns[2:]
if hay_datos(columnas, 'Aún no hay noches simuladas: en la app, pestaña Simulación, pulsa "Calcular al instante" con cada estrategia y vuelve a exportar.'):
    t = simulacion['minute'].to_numpy(dtype=float) * 60
    filas = []
    for columna in columnas:
        energia = energia_euler(np.append(t, t[-1] + 60), np.append(simulacion[columna].to_numpy(dtype=float), 0))
        filas.append({'Noche simulada': columna, 'Energía Euler (Wh)': round(energia[-1], 1)})
    display(pd.DataFrame(filas))

**Preguntas de análisis — Sección 3**
1. ¿Coincide la energía calculada con Euler con la que muestra la app? ¿A qué se deben las pequeñas diferencias?
2. ¿Qué pasaría con el error del método de Euler si el ESP32 enviara una lectura cada 5 s en lugar de cada 0,5 s?
3. En la simulación de la estrategia permanente se esperan 7 200 Wh (6 postes × 100 W × 12 h). ¿Se cumple?

---
## Sección 4 — Modelo 3: grafo de calles y Dijkstra

Las calles de la maqueta forman un **grafo no dirigido y ponderado** $G = (V, E)$:
- $V$ = postes A1, A2, A3, B1, B2, B3.
- $E$ = tramos de calle, cada uno de 30 m.

Cuando un PIR detecta a una persona, **Dijkstra** da la distancia desde la entrada hasta cada poste. Con $t_i = d_i / v$ ($v = 1{,}4$ m/s) sabemos cuándo llega la persona, y el poste se enciende 10 s antes. Esa es la **ola de luz**.

In [ ]:
calles = [('A1', 'A2', 30), ('A2', 'A3', 30), ('A2', 'B1', 30), ('B1', 'B2', 30), ('B2', 'B3', 30)]
G = nx.Graph()
G.add_weighted_edges_from(calles)
posiciones = {'A1': (0, 2), 'A2': (2, 2), 'A3': (4, 2), 'B1': (2, 1), 'B2': (2, 0), 'B3': (2, -1)}

fig, ax = plt.subplots(figsize=(6, 4))
nx.draw_networkx(G, posiciones, node_color='#ffb340', node_size=700, edgecolors='black', font_weight='bold', ax=ax)
nx.draw_networkx_edge_labels(G, posiciones, edge_labels={(u, v): f'{w} m' for u, v, w in calles}, ax=ax)
ax.set_title('Grafo de calles de la maqueta'); ax.axis('off'); plt.show()

print(f'n = |V| = {G.number_of_nodes()}    m = |E| = {G.number_of_edges()}')
for origen in ['A1', 'B3']:
    D, pred, tabla = dijkstra_tabla(G, origen)
    print(f'\nDijkstra desde la entrada {origen}:')
    display(tabla)
    print('Ruta hasta el poste más lejano:', ' → '.join(reconstruir(pred, max(D, key=D.get))))
    print('Verificación networkx:', dict(nx.single_source_dijkstra_path_length(G, origen)))

In [ ]:
VELOCIDAD = 1.4
ANTICIPACION = 10
RETENCION = 30

D, pred, tabla = dijkstra_tabla(G, 'A1')
plan = pd.DataFrame({
    'Distancia (m)': {poste: D[poste] for poste in posiciones},
    'Llegada t_i (s)': {poste: D[poste] / VELOCIDAD for poste in posiciones},
})
plan['Activa desde (s)'] = (plan['Llegada t_i (s)'] - ANTICIPACION).clip(lower=0)
plan['Retención desde (s)'] = plan['Llegada t_i (s)']
plan['Vuelve a Base (s)'] = plan['Llegada t_i (s)'] + RETENCION
display(plan.round(1))

**Preguntas de análisis — Sección 4**
1. ¿Por qué el grafo es **no dirigido**? ¿Qué cambiaría si una calle fuera de un solo sentido?
2. Al entrar por A1 se encienden también A3 y la rama B. ¿Por qué el sistema enciende **las dos ramas** en la bifurcación A2?
3. ¿Qué pasaría con la ola de luz si la persona camina a 0,8 m/s en lugar de 1,4 m/s?

---
## Sección 5 — Modelo 1: máquina de estados de cada luminaria

Cada poste está siempre en uno de cuatro estados:

| Estado | Brillo | Cuándo |
|---|---|---|
| Apagada (`Off`) | 0 % | Es de día (LDR > 40 %) |
| Base (`Base`) | 20 % | Es de noche y no viene nadie |
| Activa (`Active`) | 100 % | La persona se acerca (10 s antes de llegar) |
| Retención (`Hold`) | 100 % | La persona pasó; dura 30 s y vuelve a Base |

Con las lecturas reales contamos cuánto tiempo pasó cada poste en cada estado y cuántas veces se encendió.

In [ ]:
POSTES = ['A1', 'A2', 'A3', 'B1', 'B2', 'B3']
if hay_datos(lecturas, 'Aún no hay lecturas de la maqueta para analizar los estados.'):
    adaptativa = lecturas[(lecturas['strategy'] == 'adaptive') & (lecturas['mode'] == 'auto')]
    if hay_datos(adaptativa, 'Aún no hay noches con la estrategia adaptativa.'):
        estados = adaptativa['states'].str.split(';', expand=True)
        estados.columns = POSTES
        proporcion = estados.apply(lambda columna: columna.value_counts(normalize=True)).fillna(0).T * 100
        display(proporcion.round(1))
        proporcion.plot(kind='bar', stacked=True, figsize=(8, 4), color={'Base': '#5e5ce6', 'Active': '#ffb340', 'Hold': '#ff9f0a', 'Off': '#8e8e93'})
        plt.ylabel('% del tiempo'); plt.title('Tiempo en cada estado (estrategia adaptativa)'); plt.grid(alpha=0.3, axis='y'); plt.show()
        encendidos = {poste: int(((estados[poste] == 'Active') & (estados[poste].shift() == 'Base')).sum()) for poste in POSTES}
        print('Veces que cada poste pasó de Base a Activa:', encendidos)

**Preguntas de análisis — Sección 5**
1. ¿Qué poste pasó más tiempo encendido? ¿Coincide con lo que predice el grafo (el que está en la bifurcación)?
2. Si se aumenta el tiempo de retención de 30 s a 60 s, ¿qué estado ganaría tiempo y qué pasaría con la energía?

---
## Sección 6 — Comparación de estrategias

El **ahorro** de cada estrategia se calcula frente a la iluminación permanente, con la potencia media de las noches (así se pueden comparar noches de distinta duración):

$$ahorro = 1 - \frac{\bar{P}_{estrategia}}{\bar{P}_{permanente}}$$

Se calcula por separado con las noches de la **maqueta** y con las de la **simulación**.

In [ ]:
if hay_datos(resumen, 'Aún no hay noches registradas.'):
    limpias = resumen[resumen['strategy'].isin(NOMBRES)]
    medias = limpias.groupby(['source', 'strategy'])['averagePower'].mean().unstack('strategy')
    ahorro = pd.DataFrame(index=medias.index)
    for estrategia in NOMBRES:
        if estrategia in medias and 'permanent' in medias:
            ahorro[NOMBRES[estrategia]] = 100 * (1 - medias[estrategia] / medias['permanent'])
    print('Ahorro frente a la permanente (%):')
    display(ahorro.round(1))
    ahorro.T.plot(kind='bar', figsize=(8, 4), color=['#ffb340', '#5e5ce6'][:len(ahorro)])
    plt.ylabel('Ahorro (%)'); plt.title('Ahorro por estrategia'); plt.xticks(rotation=0); plt.grid(alpha=0.3, axis='y'); plt.show()

if hay_datos(simulacion.columns[2:], 'Aún no hay noches simuladas.'):
    fig, ax = plt.subplots(figsize=(10, 4))
    for columna in simulacion.columns[2:]:
        estrategia = columna.split('_')[0]
        ax.step(simulacion['minute'] / 60 + 18, simulacion[columna], where='post', color=COLORES.get(estrategia), label=columna)
    ax.set_xlabel('Hora de la noche (18 = 18:00)'); ax.set_ylabel('Potencia (W)')
    ax.set_title('Potencia de las noches simuladas'); ax.grid(alpha=0.3); ax.legend(); plt.show()

**Preguntas de análisis — Sección 6**
1. ¿Qué estrategia ahorra más energía? ¿Mantiene un nivel de servicio del 100 %?
2. ¿Por qué el ahorro de la programada es exactamente 25 % en la simulación?
3. ¿Por qué el ahorro adaptativo medido en la maqueta puede ser menor que el simulado? (Pista: la ola de luz de la maqueta se ve a ritmo humano dentro de una noche de 1 minuto.)

### Conclusiones
Escribe aquí las conclusiones del grupo: cumplimiento de los objetivos, qué tan bien representan los modelos a la maqueta y qué mejorarías.